In [3]:
import mysql.connector
import time
import cv2
import pickle
import os
from datetime import datetime
from win32com.client import Dispatch
from sklearn.neighbors import KNeighborsClassifier


In [4]:


# Fungsi untuk menghubungkan dan menyimpan data ke database MySQL
def initialize_database():
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="",
            database="authorized_user"
        )
    except mysql.connector.Error as err:
        print(f"Error: {err}")

def save_to_database(name, timestamp, date):
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="",
            database="authorized_user"
        )
        cursor = conn.cursor()
        print(f"Saving to database: Name: {name}, Timestamp: {timestamp}, Date: {date}")
        cursor.execute('INSERT INTO absen (name, time, date) VALUES (%s, %s, %s)', (name, timestamp, date))
        conn.commit()
        conn.close()
    except mysql.connector.Error as err:
        print(f"Error: {err}")

# Initialize the database
initialize_database()

# Fungsi untuk text-to-speech
def speak(str1):
    try:
        speak = Dispatch("SAPI.SpVoice")
        speak.Speak(str1)
    except Exception as e:
        print(f"Text-to-speech error: {e}")

# Background frame untuk tampilan
imgBackground = cv2.imread("background.png")

def detect_face():
    def draw_boundary(img, classifier, scaleFactor, minNeighbors, color, text, clf, db_conn):
        gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        features = classifier.detectMultiScale(gray_image, scaleFactor, minNeighbors)

        coords = []

        for (x, y, w, h) in features:
            cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
            try:
                id, pred = clf.predict(gray_image[y:y + h, x:x + w])
                confidence = int(100 * (1 - pred / 300))

                # Fetch the name from the database using the id
                cursor = db_conn.cursor()
                cursor.execute("SELECT name FROM siswa WHERE id=%s", (id,))
                result = cursor.fetchone()
                name = result[0] if result else "UNKNOWN"

                if confidence > 74:
                    cv2.putText(img, name, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
                else:
                    cv2.putText(img, "UNKNOWN", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 1, cv2.LINE_AA)

            except Exception as e:
                print(f"Error recognizing face: {e}")
                cv2.putText(img, "ERROR", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 1, cv2.LINE_AA)

            coords = [x, y, w, h]
        return coords

    def recognize(img, clf, faceCascade, db_conn):
        coords = draw_boundary(img, faceCascade, 1.1, 10, (255, 255, 255), "Face", clf, db_conn)
        return img

    faceCascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.read("classifier.xml")

    # Establish database connection once for use throughout the face detection process
    db_conn = mysql.connector.connect(
        host="localhost",
        user="root",
        passwd="",
        database="authorized_user"
    )

    video_capture = cv2.VideoCapture(0)

    while True:
        ret, img = video_capture.read()
        img = recognize(img, clf, faceCascade, db_conn)
        cv2.imshow("Face Detection", img)

        k = cv2.waitKey(1)
        
        if k == ord('o'):  # Press 'O' to save attendance
            timestamp = str(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
            date = str(datetime.now().strftime('%Y-%m-%d'))
            # Getting the name of the recognized person
            cursor = db_conn.cursor()
            cursor.execute("SELECT name FROM siswa WHERE id=%s", (id,))
            result = cursor.fetchone()
            name = result[0] if result else "UNKNOWN"
            
            # Save to database
            save_to_database(name, timestamp, date)
            speak("Attendance Taken.")
            time.sleep(5)
        if k == ord('q'):  # Press 'Q' to quit
            break

    video_capture.release()
    cv2.destroyAllWindows()
